In [8]:
!git clone https://github.com/tdtrinh11/ViMedNer.git

Cloning into 'ViMedNer'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 33 (delta 10), reused 21 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 1.40 MiB | 9.72 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
!pip install sklearn-crfsuite nltk

In [17]:
import os
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from nltk.tag import HiddenMarkovModelTagger

# 1. Hàm đọc dữ liệu (Dùng chung cho cả 2 mô hình)
def load_vimedner_data(file_path):
    sentences = []
    current_sentence = []
    if not os.path.exists(file_path):
        return []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue
            parts = line.split()
            if len(parts) >= 2:
                word = parts[0]
                tag = parts[-1]
                current_sentence.append((word, tag))
    if current_sentence:
        sentences.append(current_sentence)
    return sentences

# Đường dẫn dữ liệu
data_dir = 'ViMedNer/data'
train_sents = load_vimedner_data(os.path.join(data_dir, 'train.txt'))
test_sents = load_vimedner_data(os.path.join(data_dir, 'test.txt'))

# --- PHẦN 1: HIDDEN MARKOV MODEL (HMM) --- [cite: 224, 288]

# Chuẩn bị dữ liệu định dạng NLTK: list of list of (word, tag)
train_hmm = [[(w, t) for w, t in sent] for sent in train_sents]
test_hmm = [[(w, t) for w, t in sent] for sent in test_sents]

print("Đang huấn luyện HMM...")
hmm_model = HiddenMarkovModelTagger.train(train_hmm)

# Dự đoán chuỗi nhãn bằng thuật toán Viterbi [cite: 405, 409]
y_pred_hmm = []
y_test_flat = []

for sent in test_sents:
    words = [word for word, tag in sent]
    # Thuật toán Viterbi tìm chuỗi nhãn t_1...t_n tối ưu [cite: 366, 383, 778]
    tagged_sent = hmm_model.tag(words)
    y_pred_hmm.append([tag for word, tag in tagged_sent])
    y_test_flat.append([tag for word, tag in sent])

# Lấy danh sách nhãn để đánh giá (loại bỏ 'O') [cite: 214, 215]
labels_hmm = list(hmm_model._states)
if 'O' in labels_hmm: labels_hmm.remove('O')

print("\n--- KẾT QUẢ ĐÁNH GIÁ HMM ---")
print(metrics.flat_classification_report(y_test_flat, y_pred_hmm, labels=labels_hmm))

# --- PHẦN 2: CONDITIONAL RANDOM FIELDS (CRF) --- [cite: 573, 583]

def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'suffix_3': word[-3:],
        'suffix_2': word[-2:],
    }
    if i > 0:
        features['prev_word'] = sent[i-1][0].lower()
    else:
        features['BOS'] = True
    if i < len(sent)-1:
        features['next_word'] = sent[i+1][0].lower()
    else:
        features['EOS'] = True
    return features

def prepare_crf_data(sentences):
    X = [[word2features(s, i) for i in range(len(s))] for s in sentences]
    y = [[label for token, label in s] for s in sentences]
    return X, y

X_train, y_train = prepare_crf_data(train_sents)
X_test, y_test = prepare_crf_data(test_sents)

print("\nĐang huấn luyện CRF...")
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

y_pred_crf = crf.predict(X_test)
labels_crf = list(crf.classes_)
if 'O' in labels_crf: labels_crf.remove('O')

print("\n--- KẾT QUẢ ĐÁNH GIÁ CRF ---")
print(metrics.flat_classification_report(y_test, y_pred_crf, labels=labels_crf))

Đang huấn luyện HMM...

--- KẾT QUẢ ĐÁNH GIÁ HMM ---
                       precision    recall  f1-score   support

           B-ten_benh       0.73      0.80      0.76      1822
           I-ten_benh       0.77      0.85      0.81      4448
   B-trieu_chung_benh       0.53      0.55      0.54       712
   I-trieu_chung_benh       0.47      0.54      0.50      1456
B-bien_phap_chan_doan       0.51      0.58      0.54       308
I-bien_phap_chan_doan       0.48      0.55      0.51      1034
 B-bien_phap_dieu_tri       0.64      0.49      0.55       632
 I-bien_phap_dieu_tri       0.49      0.46      0.47      1716
   B-nguyen_nhan_benh       0.27      0.26      0.27       276
   I-nguyen_nhan_benh       0.22      0.33      0.26      1014

            micro avg       0.59      0.65      0.61     13418
            macro avg       0.51      0.54      0.52     13418
         weighted avg       0.60      0.65      0.62     13418


Đang huấn luyện CRF...

--- KẾT QUẢ ĐÁNH GIÁ CRF ---
        